# einops-rearrange — faded example 3: Space-to-depth: fold spatial blocks into channels

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-rearrange`. The last cell reports your progress on the `Einops: Rearrange` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-rearrange`**, which bridges to the bank subtopic `Einops: Rearrange` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Space-to-depth is the inverse of pixel shuffle: it turns `(b, c, h*r, w*r)` into `(b, c*r*r, h, w)`, packing each `r×r` spatial block into the channel axis. It needs **decomposition** of both spatial axes (`(h p1)`, `(w p2)`) bound by kwargs, a **reorder**, and a **composition** `(c p1 p2)` on the channel axis.

## Faded exercise 3

### Faded — space-to-depth downsample

Given `x` of shape `(b, c, H, W)` with `H` and `W` divisible by block size `r`, complete `space_to_depth(x, r)` to return `(b, c*r*r, H//r, W//r)` with a single `rearrange`. Everything but the pattern (and its kwargs) is filled in.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def space_to_depth(x: Tensor, r: int) -> Tensor:
    # Decompose H -> (h p1) and W -> (w p2), then fold p1,p2 into channels.
    return rearrange(x, 'b c (h p1) (w p2) -> b (c p1 p2) h w', p1=r, p2=r)


np.random.seed(0)
t.manual_seed(0)
x = t.randn(2, 3, 8, 10)  # H=8, W=10, r=2 -> (2, 12, 4, 5)
out = space_to_depth(x, r=2)
print('output shape:', tuple(out.shape))


def _test():
    t.manual_seed(42)
    b, c, H, W, r = 2, 3, 8, 10, 2
    x = t.randn(b, c, H, W)
    got = space_to_depth(x, r)
    assert got.shape == (b, c * r * r, H // r, W // r), f'wrong shape: {tuple(got.shape)}'
    # Independent reference via explicit reshape/permute.
    ref = x.reshape(b, c, H // r, r, W // r, r)
    ref = ref.permute(0, 1, 3, 5, 2, 4).reshape(b, c * r * r, H // r, W // r)
    assert t.equal(got, ref), 'values do not match reshape/permute reference'
    assert got.numel() == x.numel(), 'element count changed'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def space_to_depth(x: Tensor, r: int) -> Tensor:
    # Decompose H -> (h p1) and W -> (w p2), then fold p1,p2 into channels.
    return rearrange(x, 'b c (h p1) (w p2) -> b (c p1 p2) h w', p1=r, p2=r)


np.random.seed(0)
t.manual_seed(0)
x = t.randn(2, 3, 8, 10)  # H=8, W=10, r=2 -> (2, 12, 4, 5)
out = space_to_depth(x, r=2)
print('output shape:', tuple(out.shape))
```
</details>